## 1. Load Data

In [ ]:
import pandas as pd
import torch
import numpy as np
import unicodedata
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict

print("Loading datasets...")
word_level_df = pd.read_csv('../data/Amirim_Project_Submission/translated_podcast_transcript_filtered.csv')

with open("../data/podcast_sentences.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()
sentences = [line.strip().split(',', 1)[1] for line in lines[1:] if ',' in line]

print(f"Target words : {len(word_level_df)}")
print(f"Sentences    : {len(sentences)}")

## 2. Helpers

In [25]:
def normalize(text):
    text = unicodedata.normalize('NFC', str(text))
    # Added: strip apostrophes so "didn't" -> "didnt", "Wikipedia's" -> "wikipedias"
    text = text.replace("\u2019", "").replace("'", "").replace("`", "")
    return text.strip(' .,!?"()-:;[]{}').lower()

def sentence_contains_components(sentence_lower, components):
    """Check if all phrase components appear consecutively in the sentence."""
    # Added: replace hyphens with spaces so "self-satisfied" -> ["self", "satisfied"]
    sentence_lower = sentence_lower.replace("-", " ")
    words = sentence_lower.split()
    words_norm = [normalize(w) for w in words]
    for i in range(len(words_norm) - len(components) + 1):
        if all(words_norm[i+j] == components[j] for j in range(len(components))):
            return True
    return False

def get_sentence_tokens(sentence, tokenizer, model):
    encoded = tokenizer(sentence, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**encoded)
    token_embeddings = outputs.last_hidden_state.squeeze(0)
    word_ids = encoded.word_ids()
    input_ids = encoded['input_ids'][0]

    word_vectors = {}
    word_token_ids = {}
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id not in word_vectors:
            word_vectors[word_id] = []
            word_token_ids[word_id] = []
        word_vectors[word_id].append(token_embeddings[idx].numpy())
        word_token_ids[word_id].append(input_ids[idx].item())

    tokens = []
    for word_id in sorted(word_vectors.keys()):
        avg_vector = np.mean(word_vectors[word_id], axis=0)
        raw_text = tokenizer.decode(word_token_ids[word_id])
        norm_text = normalize(raw_text)
        if norm_text:
            tokens.append({"text": norm_text, "vector": avg_vector})
    return tokens

## 3. Assign Words to Sentences

In [ ]:
print("Assigning target words to sentences...")

word_to_sentence = {}
unassignable = []
assignment_counts = defaultdict(lambda: defaultdict(int))

sent_idx = 0
for wi in range(len(word_level_df)):
    target_raw = str(word_level_df.iloc[wi]['en']).strip().lower()
    components = [c.strip() for c in target_raw.split('_')]
    target_key = '_'.join(components)

    found = False
    for lookahead in range(min(6, len(sentences) - sent_idx)):
        candidate_idx = sent_idx + lookahead
        
        sent_words = [normalize(w) for w in sentences[candidate_idx].replace("-", " ").replace("%", " percent ").split()]        
        
        capacity = 0
        i = 0
        while i <= len(sent_words) - len(components):
            if all(sent_words[i+j] == components[j] for j in range(len(components))):
                capacity += 1
                i += len(components)
            else:
                i += 1
        
        already_assigned = assignment_counts[candidate_idx][target_key]
        
        if capacity > already_assigned:
            word_to_sentence[wi] = candidate_idx
            assignment_counts[candidate_idx][target_key] += 1
            sent_idx = candidate_idx
            found = True
            break

    if not found:
        unassignable.append(wi)
        word_to_sentence[wi] = None

assigned = sum(1 for v in word_to_sentence.values() if v is not None)
print(f"Assigned : {assigned} / {len(word_level_df)}")
print(f"Dropped  : {len(unassignable)}")

print("\nFirst 10 assignments:")
for wi in range(10):
    si = word_to_sentence[wi]
    sent_preview = sentences[si][:70] if si is not None else "UNASSIGNED"
    print(f"  [{wi:3d}] '{word_level_df.iloc[wi]['en']}'  -> sent {si}: '{sent_preview}'")

In [ ]:
# --- 3b. RESCUE: full scan for unassigned words, no pointer ---
print(f"Rescuing {len(unassignable)} unassigned words via full scan...\n")

# Build time map from successfully assigned words
sentence_time_map = {}
for wi, si in word_to_sentence.items():
    if si is None:
        continue
    t_start = word_level_df.iloc[wi]['start']
    t_end = word_level_df.iloc[wi]['end']
    if si not in sentence_time_map:
        sentence_time_map[si] = [t_start, t_end]
    else:
        sentence_time_map[si][0] = min(sentence_time_map[si][0], t_start)
        sentence_time_map[si][1] = max(sentence_time_map[si][1], t_end)

rescued = 0
still_unassignable = []

for wi in unassignable:
    target_raw = str(word_level_df.iloc[wi]['en']).strip().lower()
    components = [c.strip() for c in target_raw.split('_')]
    target_key = '_'.join(components)
    t = word_level_df.iloc[wi]['start']
    
    candidates = []
    for si in range(len(sentences)):
        sent_words = [normalize(w) for w in 
                      sentences[si].replace("-", " ")
                                   .replace("%", " percent ")
                                   .split()]
        capacity = 0
        i = 0
        while i <= len(sent_words) - len(components):
            if all(sent_words[i+j] == components[j] for j in range(len(components))):
                capacity += 1
                i += len(components)
            else:
                i += 1
        
        remaining = capacity - assignment_counts[si][target_key]
        if remaining > 0:
            if si in sentence_time_map:
                sent_t = sentence_time_map[si][0]
                proximity = abs(sent_t - t)
            else:
                proximity = 9999
            candidates.append((proximity, si))
    
    if candidates:
        candidates.sort()
        best_si = candidates[0][1]
        word_to_sentence[wi] = best_si
        assignment_counts[best_si][target_key] += 1
        rescued += 1
    else:
        still_unassignable.append(wi)

print(f"Rescued via full scan : {rescued}")
print(f"True unassignable     : {len(still_unassignable)}")
unassignable = still_unassignable

if still_unassignable:
    print(f"\nWords with no matching sentence anywhere:")
    for wi in still_unassignable:
        row = word_level_df.iloc[wi]
        print(f"  [{wi:4d}] t={row['start']:.1f}s  '{row['en']}'")

## 4. Load Model

In [ ]:
print("Loading XLM-RoBERTa...")
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModel.from_pretrained('xlm-roberta-base')
model.eval()

## 5. Extraction Loop

In [ ]:
print("Extracting embeddings...\n")

final_aligned_embeddings = []
matched_word_indices = []
dropped_word_indices = []

sentence_token_cache = {}

def get_tokens_cached(si):
    if si not in sentence_token_cache:
        sentence_token_cache[si] = get_sentence_tokens(sentences[si], tokenizer, model)
    return sentence_token_cache[si]

words_per_sentence = defaultdict(list)
for wi, si in word_to_sentence.items():
    if si is not None:
        words_per_sentence[si].append(wi)

for si in sorted(words_per_sentence.keys()):
    target_word_indices = sorted(words_per_sentence[si])
    sentence_tokens = get_tokens_cached(si)
    n_tokens = len(sentence_tokens)

    consumed_positions = set()
    sentence_pointer = 0

    for word_idx in target_word_indices:
        target_raw = str(word_level_df.iloc[word_idx]['en']).strip().lower()
        target_components = [c.strip() for c in target_raw.split('_')]
        phrase_len = len(target_components)

        def try_match_at(start):
            if start + phrase_len > n_tokens:
                return False
            if set(range(start, start + phrase_len)) & consumed_positions:
                return False
            return all(
                sentence_tokens[start + i]['text'] == target_components[i]
                for i in range(phrase_len)
            )

        # Pass 1: forward from current pointer
        matched_at = None
        for start in range(sentence_pointer, n_tokens):
            if try_match_at(start):
                matched_at = start
                break

        # Pass 2: backward (handles repeated words)
        if matched_at is None:
            for start in range(0, sentence_pointer):
                if try_match_at(start):
                    matched_at = start
                    break

        if matched_at is not None:
            phrase_vectors = [sentence_tokens[matched_at + i]['vector']
                              for i in range(phrase_len)]
            final_aligned_embeddings.append(np.mean(phrase_vectors, axis=0))
            matched_word_indices.append(word_idx)
            for i in range(phrase_len):
                consumed_positions.add(matched_at + i)
            if matched_at >= sentence_pointer:
                sentence_pointer = matched_at + phrase_len
        else:
            dropped_word_indices.append(word_idx)

## 6. Report

In [ ]:
print("=" * 55)
print("FINAL ALIGNMENT REPORT")
print("=" * 55)

unassigned_count = len(unassignable)
extracted_count = len(final_aligned_embeddings)
dropped_in_extraction = len(dropped_word_indices)

print(f"Total target words       : {len(word_level_df)}")
print(f"Assigned to a sentence   : {len(word_level_df) - unassigned_count}")
print(f"Unassigned (no sentence) : {unassigned_count}")
print(f"Successfully matched     : {extracted_count}")
print(f"Dropped in extraction    : {dropped_in_extraction}")
print(f"Total accounted for      : {extracted_count + dropped_in_extraction + unassigned_count}")
print(f"Match rate (of total)    : {extracted_count/len(word_level_df)*100:.1f}%")

if dropped_word_indices:
    print(f"\nDropped in extraction:")
    for idx in dropped_word_indices:
        row = word_level_df.iloc[idx]
        si = word_to_sentence.get(idx)
        print(f"  [{idx:4d}] '{row['en']}'  -> sent {si}: '{sentences[si][:60] if si else 'NONE'}'")

if unassignable:
    print(f"\nFirst 20 unassigned words (no sentence found):")
    for wi in unassignable[:20]:
        row = word_level_df.iloc[wi]
        print(f"  [{wi:4d}] t={row['start']:.1f}s  '{row['en']}'")

## 7. Save

In [ ]:
if len(final_aligned_embeddings) > 0:
    embeddings_df = pd.DataFrame(final_aligned_embeddings)
    out_emb = '../data/processed/en_contextual_aligned_embeddings.csv'
    out_idx = '../data/processed/en_contextual_matched_indices.csv'
    embeddings_df.to_csv(out_emb, index=False)
    pd.DataFrame({'original_word_idx': matched_word_indices}).to_csv(out_idx, index=False)
    print(f"Saved embeddings -> {out_emb}  shape: {embeddings_df.shape}")
    print(f"Saved index map  -> {out_idx}")

    emb = pd.read_csv('../data/processed/en_contextual_aligned_embeddings.csv')
    idx = pd.read_csv('../data/processed/en_contextual_matched_indices.csv')

    print(f"Embeddings shape : {emb.shape}")       # should be (1441, 768)
    print(f"Index map shape  : {idx.shape}")        # should be (1441, 1)
    print(f"Index range      : {idx['original_word_idx'].min()} - {idx['original_word_idx'].max()}")
    print(f"Any NaN in embeddings: {emb.isnull().any().any()}")
    print(f"\nFirst 5 indices: {idx['original_word_idx'].tolist()[:5]}")
    print(f"Sample embedding row 0 (first 5 dims): {emb.iloc[0, :5].tolist()}")